# Adaptive CPU Auto-Scaling Using Hybrid Neural Networks and Continual Learning
## Research Implementation Notebook (`research_imp`)

**Research Topic:** Adaptive CPU Auto-Scaling Using Hybrid Neural Networks (LSTM + MLP)
and Continual Learning (EWC + ER) in Kubernetes Orchestration

**Dataset used here:** prebuilt Alibaba time-series (`alibaba_timeseries_full.csv`)

---

### Research Objectives (Proposal Section 2.3)
1. Forecast CPU demand using a **hybrid neural model** (LSTM for temporal patterns + MLP for static features)
2. Use **EWC** and **Experience Replay** to mitigate catastrophic forgetting under changing workloads
3. Improve Kubernetes scaling decisions beyond reactive threshold-based (HPA) methods

### Notebook Structure
- **Phase 0** — Configuration and execution controls
- **Phase 1** — Data loading and feature engineering from preprocessed time-series
- **Phase 2** — Sequence preparation and chunk setup
- **Phase 3** — Continual learning training (Hybrid + EWC + ER)
- **Phase 4** — Evaluation and baseline comparison
- **Phase 5** — Optional validation blocks and final artefacts


In [ ]:
# Import runner API from the Python implementation file.
from pathlib import Path
from research_imp import ExperimentConfig, run_experiment, STAGE_ORDER

print('Imports loaded.')
print('Available stages for run_until:')
print(', '.join(STAGE_ORDER))

---
## Phase 0 — Configuration and Execution Controls

Use the flags below to choose **how far** the notebook should execute and which
validation blocks should run.

- `run_until='dashboard'` runs the full pipeline.
- Example partial run: `run_until='train'` stops after training.
- You can independently disable specific validation blocks with booleans.
- Resume mode uses `experiment_manifest.json` + config signature matching.


In [ ]:
# Main configuration for this notebook run.
# Edit these values before running the pipeline cell.
cfg = ExperimentConfig(
    preprocessed_path='alibaba_timeseries_full.csv',
    output_dir='.',
    interval=300,
    history_length=24,
    forecast_steps=6,
    n_chunks=4,
    epochs=10,
    batch_size=32,
    ewc_lambda=100.0,
    replay_memory=500,
    replay_ratio=0.3,
    roll_window=12,
    sla_tol=0.15,
    seed=42,
    run_until='dashboard',
    run_multi_seed=True,
    run_ablation=True,
    run_forgetting=True,
    run_cross_app=True,
    run_sensitivity=True,
    run_sla_analysis=True,
    run_dashboard=True,
    resume=True,
    force_retrain=False,
    force_baselines=False,
    force_validation=False,
    refresh_all=False,
    fast_mode=False,
)

print('Notebook configuration loaded:')
print(cfg)

---
## Run Pipeline

This executes the full research pipeline through the selected stage (`run_until`)
and prints progress/timing in the output.


In [ ]:
# Execute the pipeline and store all outputs in the `results` dictionary.
results = run_experiment(cfg)

---
## Phase 1 — Core Results

The tables below summarize chunk-wise continual learning performance and
baseline comparison on the final chunk.


In [ ]:
import pandas as pd

# Show chunk-level metrics if available in this run scope.
chunk_metrics = results.get('chunk_results')
if chunk_metrics:
    display(pd.DataFrame(chunk_metrics))
else:
    print('Chunk metrics not available (run_until may be before training).')

In [ ]:
# Show baseline comparison table if available.
baseline_df = results.get('results_df')
if baseline_df is not None:
    display(baseline_df.round(4))
else:
    print('Baseline table not available (run_until may be before baselines).')

---
## Phase 2 — Execution Summary

This section displays which stages were executed in this notebook run.


In [ ]:
print('Run-until target:', results.get('run_until'))
print('Stage statuses:')
for k, v in results.get('stage_status', {}).items():
    print(f'  - {k}: {v}')

print('Baseline statuses:')
for k, v in results.get('baseline_status', {}).items():
    print(f'  - {k}: {v}')

manifest = results.get('manifest', {})
print('Manifest signature present:', manifest.get('signature') is not None)
print('Executed stages:')
for i, st in enumerate(results.get('executed_stages', []), start=1):
    print(f'  {i:02d}. {st}')

---
## Phase 3 — Generated Artefacts

The pipeline writes model/scaler files, result tables, and figures to `output_dir`.


In [ ]:
# Check expected output files in the configured output directory.
out_dir = Path(cfg.output_dir)
artifact_files = [
    'hybrid_model_ewc_er.keras',
    'temporal_scaler.pkl',
    'static_scaler.pkl',
    'target_scaler.pkl',
    'chunk_metrics.csv',
    'baseline_results.csv',
    'chunk_metrics.json',
    'baseline_results.json',
    'evaluation_results.png',
    'validation_dashboard.png',
]

for name in artifact_files:
    p = out_dir / name
    print(f'{name:<30} -> {"OK" if p.exists() else "MISSING"}')

---
## Phase 4 — Notes for Dissertation Reporting

- Use `chunk_metrics.csv` for continual-learning progression tables.
- Use `baseline_results.csv` for final comparison and percentage improvement.
- Use `evaluation_results.png` and `validation_dashboard.png` as report figures.
- If needed, rerun with a different `run_until` value to execute only specific phases.
